## Sesion 3 - Pruebas unitarias con pytest

**Bootcamp:** Python PY4  
**Modulo 4:** Big Data y MLOps  
**Taller de Calidad: Pruebas Unitarias y Documentación Profesional**

---

Una funcion documentada esta bien, pero lo que realmente da confianza de que el codigo funciona (y que sigue funcionando despues de un cambio) son las pruebas automaticas.  <br><br>
Hoy escribimos `tests/test_procesamiento.py`, corremos pytest, y cerramos el flujo completo: estructura, documentacion, pruebas, y entrega final en un `.zip`.

## Objetivos de la sesion

Al final de esta sesion vas a poder:

- Explicar que es una prueba unitaria y por que debe ser independiente de otras pruebas.
- Escribir pruebas con `pytest`, usando `assert` y `pytest.raises`.
- Probar `calcular_metricas_ventas` con datos conocidos y con un dataframe vacio.
- Correr la suite de pruebas y leer su resultado.

## Parte 0: Reconectando con el proyecto

Retomamos el mismo proyecto de las sesiones anteriores. Montamos Drive y nos ubicamos de nuevo en la carpeta del proyecto.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

PROJECT_NAME = "taller_calidad"
PROJECT_PATH = f"/content/drive/MyDrive/{PROJECT_NAME}"

os.chdir(PROJECT_PATH)

print("Trabajando en:", os.getcwd())

Trabajando en: /content/drive/MyDrive/taller_calidad


Confirmemos que el historial de las dos sesiones anteriores sigue ahi. Deberian aparecer dos commits.

In [4]:
!git log --oneline

417a95f (HEAD -> master) Documenta calcular_metricas_ventas con docstring estilo NumPy
65f550c Agrega archivo inicial src/procesamiento.py
53883aa Estructura inicial del proyecto: carpetas, requirements y gitignore


Y confirmemos que `calcular_metricas_ventas` sigue en el archivo tal como la dejamos en la sesion 2.

In [5]:
!cat src/procesamiento.py # cat = "ensename todo lo que hay dentro de src/procesamiento.py."


"""Modulo de procesamiento de metricas de ventas."""  # Describimos brevemente el objetivo del modulo

import pandas as pd  # Importamos Pandas para trabajar con DataFrames


def calcular_metricas_ventas(dataframe):
    """
    Calcula metricas clave a partir de un dataset de ventas.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Dataset de ventas con columnas ['producto', 'cantidad', 'precio_unitario'].

    Returns
    -------
    dict
        Diccionario con las llaves 'total_ventas', 'ticket_promedio' y
        'cantidad_transacciones'. Si el dataframe esta vacio, las tres
        metricas se devuelven en cero.
    """
    if dataframe.empty:  # Comprobamos si el DataFrame esta vacio (".empty" se puede usar sobre cualquier dataframe, series o inex de Pandas, it returns True or False)
        return {
            "total_ventas": 0.0,  # Si esta vacio, el total de ventas es cero
            "ticket_promedio": 0.0,  # Si esta vacio, el ticket promedio es cero

## Parte 1: Instalando pytest

Cada vez que arranca un runtime nuevo de Colab, las librerias instaladas con pip se pierden, aunque los archivos en Drive sigan ahi. El `requirements.txt` que dejamos en la sesion 1 es justo para esto: nos permite reinstalar el entorno exacto del proyecto con un solo comando.

In [6]:
!pip install -r requirements.txt -q # -r = "read", -q = "quiet"

## Parte 2: Por que escribir pruebas unitarias

Una prueba unitaria es un fragmento de codigo que verifica que una funcion especifica se comporte como se espera, de forma automatica y repetible. En proyectos de datos esto es clave: un error silencioso en un calculo (por ejemplo, un total mal calculado) puede propagarse a un reporte o a un modelo sin que nadie lo note, hasta que ya es tarde.

Usamos el framework **pytest** porque es simple: una funcion cuyo nombre empieza con `test_`, y que usa `assert` para verificar un resultado, ya es una prueba valida.

Dos principios importantes:

- **Cada prueba debe ser independiente**: el resultado de `test_a` no deberia depender de que `test_b` se haya ejecutado antes.
- **No probar contra recursos externos reales**: si una funcion necesita una base de datos o una API, en la prueba se simula (mock) ese recurso, no se conecta a el de verdad.

Veamos un ejemplo simple. Definimos una funcion y sus pruebas, y las corremos llamandolas como funciones normales.

In [7]:
import pytest

# Definimos una función que calcula el IVA sobre un monto
def calcular_iva(monto, tasa=0.13):
    if monto < 0:  # Verificamos que el monto no sea negativo
        raise ValueError("El monto no puede ser negativo")  # Generamos un error si el monto es inválido
    return round(monto * tasa, 2)  # Calculamos el IVA y redondeamos el resultado a dos decimales

# Definimos una prueba para verificar el cálculo normal
def test_calcular_iva_estandar():
    assert calcular_iva(100) == 13.0  # Comprobamos que el IVA de 100 sea 13

## si este linea no da niguno output, la prueba es exitosa
test_calcular_iva_estandar()

In [8]:
def test_calcular_iva_falla():
    assert calcular_iva(100) == 15.0  # Incorrecto: la función devuelve 13.0

#test_calcular_iva_falla()

try:
    test_calcular_iva_falla()
except AssertionError:
    print("❌ La prueba falló: el resultado no coincide con el esperado")

❌ La prueba falló: el resultado no coincide con el esperado


In [9]:
# Definimos una prueba para verificar el manejo de valores inválidos
def test_calcular_iva_error():
    with pytest.raises(ValueError):  # Comprobamos que la función genere un ValueError (“I expect the code inside this block to raise a ValueError.”)
        calcular_iva(-50)  # Probamos la función con un monto negativo
# --> "Espero que esta línea genere un error ValueError y, si lo hace, la prueba es exitosa."


test_calcular_iva_error()

Si alguna de las dos pruebas fallara, Python levantaria un `AssertionError` (o no se cumpliria el `pytest.raises`) y veriamos el error inmediatamente. Esto es lo que hace utiles a las pruebas: fallan de forma ruidosa y especifica cuando algo esta mal, en vez de fallar en silencio.

#### Ejemplo de una mala práctica: una prueba depende de otra

In [10]:
resultado = None  # Creamos una variable global con valor inicial None

def test_B():
    global resultado  # Indicamos que vamos a modificar la variable global llamada resultado (que esta a fuera de la funcion)
    resultado = 10  # Cambiamos el valor de la variable global a 10

    assert resultado == 10  # Verificamos que resultado tenga el valor esperado


def test_A():
    assert resultado == 10  # test_A utiliza el valor que fue modificado por test_B


# Si ejecutamos primero test_B y después test_A:
test_B()  # Preparamos el valor de resultado
test_A()  # Dependemos del valor que dejó test_B

Este ejemplo funciona únicamente si `test_B` se ejecuta antes que `test_A`.

El problema es que `test_A` está utilizando un estado que fue creado por otra prueba. `test_A` no puede ejecutarse de forma independiente!

**---> Cada prueba debe crear sus propios datos y preparar su propio estado.**

## Parte 3: Escribiendo `tests/test_procesamiento.py`

Vamos a escribir dos pruebas para `calcular_metricas_ventas`:

- **Test de calculo**: verifica que, con datos conocidos, el total de ventas, el ticket promedio y la cantidad de transacciones sean los esperados.
- **Test de vacio**: verifica que, si el dataframe no tiene filas, la funcion devuelva las metricas en cero en vez de lanzar un error.

Importamos la funcion directamente desde `src.procesamiento`, tal como hicimos en el notebook de la sesion 2.

In [11]:
%%writefile tests/test_procesamiento.py
"""Pruebas unitarias para src/procesamiento.py"""

import pandas as pd  # Importamos Pandas para crear DataFrames de prueba

from src.procesamiento import calcular_metricas_ventas  # Importamos la función que vamos a probar


def test_calcular_metricas_con_datos():
    # Creamos un DataFrame con datos de ventas para probar el caso normal
    df = pd.DataFrame({
        "producto": ["A", "B"],
        "cantidad": [2, 1],
        "precio_unitario": [10.0, 20.0],
    })

    resultado = calcular_metricas_ventas(df)  # Ejecutamos la función con los datos de prueba

    # Verificamos que las métricas calculadas sean las esperadas
    assert resultado["total_ventas"] == 40.0
    assert resultado["cantidad_transacciones"] == 2
    assert resultado["ticket_promedio"] == 20.0


def test_calcular_metricas_dataframe_vacio():
    # Creamos un DataFrame vacío para probar el caso sin datos
    df = pd.DataFrame(columns=["producto", "cantidad", "precio_unitario"])

    resultado = calcular_metricas_ventas(df)  # Ejecutamos la función con el DataFrame vacío

    # Verificamos que las métricas sean cero cuando no hay datos
    assert resultado["total_ventas"] == 0.0
    assert resultado["ticket_promedio"] == 0.0
    assert resultado["cantidad_transacciones"] == 0

Writing tests/test_procesamiento.py


La idea no de poder **modificar el codigo en el futuro con confianza**. Si cambiamos una función y rompemos algo, las pruebas nos ayudan a detectar el problema rápidamente.

En nuestro caso, probamos tanto el **caso normal** como un **caso límite**, como cuando el DataFrame está vacío.

Corramos la suite completa con pytest. Usamos `python -m pytest` en vez de solo `pytest`, para asegurarnos de que se use el mismo interprete de Python que tiene pandas instalado.

In [12]:
!python -m pytest tests/ -v
# python -m pytest ---> “From the notebook, ask the shell/terminal to start Python and run the pytest module on my tests.”
# -v ---> asks pytest to show more detailed information about the tests.

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/taller_calidad
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 2 items                                                              

tests/test_procesamiento.py::test_calcular_metricas_con_datos PASSED     [ 50%]
tests/test_procesamiento.py::test_calcular_metricas_dataframe_vacio PASSED [100%]

============================== 2 passed in 2.06s ===============================


Cada linea del resultado corresponde a una prueba: `PASSED` en verde si el `assert` se cumplio, `FAILED` en rojo si no. Si una prueba falla, pytest muestra ademas el valor exacto que esperaba y el que recibio, lo que ayuda mucho a encontrar el problema.

Lo que acaba de pasar:  
**“Pytest, ve a la carpeta `tests/`, encuentra las pruebas, ejecútalas todas, verifica los `assert` y muéstra los resultados.”**

```
tests/
│
├── test_procesamiento.py    ← pytest detecta este archivo
├── test_ventas.py            ← pytest detecta este archivo
├── otro_archivo.py           ← no se detecta
└── datos.py                  ← no se detecta
```


## Errores comunes

- **No aislar las pruebas**: probar una funcion que depende de una conexion real a una base de datos o a una API externa. En esos casos se usan mocks en vez de conectarse de verdad.
- **Pruebas dependientes entre si**: que `test_b` solo pase si `test_a` se ejecuto antes. Cada prueba debe poder correr sola, en cualquier orden.
- **Probar solo el camino feliz**: si `calcular_metricas_ventas` solo se prueba con datos "normales", nunca se descubre que pasa con un dataframe vacio, hasta que se rompe en produccion.

## Ejercicio: refactoriza, documenta y prueba

Se te entrega la siguiente funcion, que calcula el ROI (retorno de inversion) de una campana de marketing. Tiene un problema: no maneja el caso en que la inversion sea cero (division por cero), no tiene docstring, y no tiene pruebas.

```python
def calcular_roi(ingresos, inversion):
    roi = (ingresos - inversion) / inversion
    return roi
```

Lo que hay que hacer:

1. Refactoriza la funcion para que maneje el caso de `inversion == 0` (por ejemplo, lanzando un `ValueError` en vez de dejar que Python lance `ZeroDivisionError`).
2. Escribe el docstring en estilo NumPy.
3. Escribe al menos dos pruebas con pytest: una con un calculo exitoso, y una que verifique el manejo del error cuando `inversion` es 0.

La funcion va en un archivo nuevo, `src/marketing.py`, con sus pruebas correspondientes en `tests/test_marketing.py`.

In [13]:
%%writefile src/marketing.py

""" Modulo de metrica de marketing """

def calcular_roi(ingresos, inversion):
  """
  Calcula el retorno de inversion (ROI) de una campana.

    Parameters
    ----------
    ingresos : float
        Ingresos generados por la campana.
    inversion : float
        Monto invertido en la campana.

    Returns
    -------
    float
        El ROI, calculado como (ingresos - inversion) / inversion.

    Raises
    ------
    ValueError
        Si la inversion es igual a cero.

  """
  if inversion == 0:
    raise ValueError("la inversion no puede ser cero")
  return round((ingresos - inversion) / inversion, 3)


Writing src/marketing.py


In [16]:
%%writefile tests/test_marketing.py

"""Pruebas unitarias para src/marketing.py"""

import pytest

from src.marketing import calcular_roi

def test_calcular_roi_exitoso():
    assert calcular_roi(1500, 1000) == 0.5

def test_calcular_roi_inversion_cero():
    with pytest.raises(ValueError):
        calcular_roi(100, 0)


Overwriting tests/test_marketing.py


Corremos toda la suite de nuevo. Si todo esta correcto, ahora deberian pasar cuatro pruebas en total: las dos de `calcular_metricas_ventas` y las dos de `calcular_roi`.

In [17]:
!python -m pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/taller_calidad
plugins: typeguard-4.6.0, langsmith-0.12.5, anyio-4.15.1
collected 4 items                                                              

tests/test_marketing.py::test_calcular_roi_exitoso PASSED                [ 25%]
tests/test_marketing.py::test_calcular_roi_inversion_cero PASSED         [ 50%]
tests/test_procesamiento.py::test_calcular_metricas_con_datos PASSED     [ 75%]
tests/test_procesamiento.py::test_calcular_metricas_dataframe_vacio PASSED [100%]

============================== 4 passed in 0.58s ===============================


## Guardando el cierre del proyecto en Git

In [18]:
!git status

On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	src/marketing.py
	tests/

nothing added to commit but untracked files present (use "git add" to track)


In [20]:
!git config --global user.email "mi-correo@example.com"
!git config --global user.name "Camille"

# Verificamos que haya quedado configurado
!git config user.name
!git config user.email

Camille
mi-correo@example.com


In [21]:
!git add .
!git commit -m "Agrega pruebas unitarias y funcion calcular_roi documentada"

[master d70ef7f] Agrega pruebas unitarias y funcion calcular_roi documentada
 3 files changed, 74 insertions(+)
 create mode 100644 src/marketing.py
 create mode 100644 tests/test_marketing.py
 create mode 100644 tests/test_procesamiento.py


In [22]:
!git log --oneline

d70ef7f (HEAD -> master) Agrega pruebas unitarias y funcion calcular_roi documentada
417a95f Documenta calcular_metricas_ventas con docstring estilo NumPy
65f550c Agrega archivo inicial src/procesamiento.py
53883aa Estructura inicial del proyecto: carpetas, requirements y gitignore


## Cierre del taller

Repasemos el proyecto completo que construimos a lo largo de las tres sesiones:

- Estructura de carpetas (`data/`, `src/`, `tests/`) con `requirements.txt`, `.gitignore` y `README.md`.
- `calcular_metricas_ventas` documentada con un docstring estilo NumPy.
- `calcular_roi` refactorizada, documentada y probada como ejercicio de integracion.
- Pruebas unitarias con pytest, incluyendo casos de exito y casos limite (dataframe vacio, inversion en cero).
- Historial de Git con varios commits, y el proyecto comprimido en un `.zip` listo para entregar.
